# LinearRegressionModel — Parallel Search (Doyle set 1)
**Target:** `output` | **Left-out:** L10, L27, L21, L18 | **Features:** 86 → ~2.1M 4-feature combos

In [1]:
import sys, os, multiprocessing, time, threading
import numpy as np
import pandas as pd

# Silence PyTensor g++ warning before anything imports pymc
os.environ.setdefault('PYTENSOR_FLAGS', 'cxx=')

MODULE_DIR = r'C:\Users\edens\Desktop\DescriPytor\DescriPyTor-main\MolFeatures\M3_modeler'
if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

from modeling import LinearRegressionModel

N_CORES = multiprocessing.cpu_count()
print(f'CPU cores: {N_CORES}')

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`


CPU cores: 20


## 1. Load dataset

In [2]:
modeling_example_dir = r'C:\Users\edens\Desktop\Lab\datasets\doyle\doyle_results_set1'
csv_path = modeling_example_dir + r'\doyle_features1_new.csv'

csv_filepaths = {'features_csv_filepath': csv_path, 'target_csv_filepath': ''}

df = pd.read_csv(csv_path, encoding='utf-8')
print(f'Rows: {df.shape[0]}  |  Columns: {df.shape[1]}')
df.head(3)

Rows: 29  |  Columns: 88


,name,Frequency_Stretch_1_5,Amplitude_Stretch_1_5,dipole_x_2-1-5,dipole_y_2-1-5,dipole_z_2-1-5,total_dipole_2-1-5,dipole_x_5-4-3,dipole_y_5-4-3,dipole_z_5-4-3,...,bond_length_18-5,bond_length_18-6,bond_length_1-18,percent_buried_volume,fraction_buried_volume,buried_volume,free_volume,energy,log10_B1_B5_an,output
0,L1,1742.8658,1.350258,0.723202,-1.368471,0.008738,1.5479,0.481230,1.471121,0.005509,...,1.933626,1.933224,2.626088,0.377481,0.377481,67.793448,111.800932,8.595809,-0.603626,0.802527
1,L2,1745.7034,1.368818,0.711446,-1.299814,0.221495,1.4982,0.466279,1.410932,0.191273,...,3.338020,2.514907,3.921683,0.440889,0.440889,79.181258,100.413122,9.979237,0.140225,1.629255
2,L3,1746.2773,1.375466,0.728900,-1.253183,0.262156,1.4733,0.430487,1.389362,0.234188,...,4.811208,2.676889,4.815565,0.615979,0.615979,110.626381,68.967999,9.885357,0.331994,1.743479


## 2. Build the model

In [3]:
leave_out = ['L10', 'L27', 'L21', 'L18']

regression_model1 = LinearRegressionModel(
    csv_filepaths,
    process_method='one csv',
    y_value='output',
    leave_out=leave_out,
    min_features_num=4,
    max_features_num=4,
    metrics=None,
    return_coefficients=False,
)

# How many combinations will we search?
from math import comb
n_features = len(regression_model1.feature_names)
n_combos   = comb(n_features, 4)
print(f'Training molecules : {len(regression_model1.molecule_names)}')
print(f'Left-out           : {regression_model1.molecule_names_predict}')
print(f'Features           : {n_features}')
print(f'4-feature combos   : {n_combos:,}')

Reusing existing run directory: runs\doyle_features1_new_output_linear_20260629
Database already exists at: C:\Users\edens\Desktop\DescriPytor\DescriPyTor-main\MolFeatures\M3_modeler\runs\doyle_features1_new_output_linear_20260629\db\results_doyle_features1_new.db
Table 'regression_results' has been ensured to exist.
Processed doyle_features1_new.csv
Molecule names: from column 'name' | Target: output
Rows: 29 â 29 (dropped NaN targets: 0) | Features: 86 â 86
Training molecules : 25
Left-out           : ['L10', 'L27', 'L21', 'L18']
Features           : 86
4-feature combos   : 2,123,555


## 3. Quick speed estimate

Run 500 combos and extrapolate to the full set so you know what to expect.

In [4]:
from joblib import Parallel, delayed
from modeling import fit_and_evaluate_single_combination_regression

regression_model1.get_feature_combinations()
sample = list(regression_model1.features_combinations)[:500]

t0 = time.perf_counter()
Parallel(n_jobs=N_CORES, backend='threading')(
    delayed(fit_and_evaluate_single_combination_regression)(regression_model1, c, 0.0, False)
    for c in sample
)
elapsed = time.perf_counter() - t0

rate          = 500 / elapsed
eta_seconds   = n_combos / rate
eta_minutes   = eta_seconds / 60

print(f'Rate       : {rate:,.0f} combos/sec  ({N_CORES} threads)')
print(f'ETA (full) : {eta_minutes:.1f} min  ({n_combos:,} combos)')

Rate       : 414 combos/sec  (20 threads)
ETA (full) : 85.6 min  (2,123,555 combos)


## 4. Run search in background thread

The search uses `threading` backend — NumPy releases the GIL so all cores are used with no process-spawn cost and no MemoryError.

Run this cell and keep working. Check the status cell below at any time.

In [ ]:
bg = {'results': None, 'error': None, 'done': False, 't_start': None, 't_end': None}

def _run():
    bg['t_start'] = time.perf_counter()
    try:
        bg['results'] = regression_model1.search_models(
            top_n=20,
            n_jobs=N_CORES,
            threshold=0.77,
        )
    except Exception as e:
        bg['error'] = e
        import traceback; traceback.print_exc()
    finally:
        bg['t_end'] = time.perf_counter()
        bg['done']  = True
        elapsed = bg['t_end'] - bg['t_start']
        print(f'\n✅ Search finished in {elapsed/60:.1f} min.')

search_thread = threading.Thread(target=_run, daemon=True)
search_thread.start()
print(f'Search launched in background ({N_CORES} threads).')
print('Run the status cell below to check progress at any time.')

Search launched in background (20 threads).
Run the status cell below to check progress at any time.


Using 20 jobs for evaluation. Found 20 cores.
Loaded 0 existing results from DB.
Combos to run: 2123555, done_combos: 0
Evaluating 2123555 new combos with R2 >= 0.770...
Parallel backend: threading  (NumPy releases GIL â 20 threads)


Threshold 0.770 (threads): 100%|██████████| 2123555/2123555 [8:48:18<00:00, 66.99it/s]      


Batch DB insert failed (No module named 'pandas.io.formats.csvs'); falling back to per-row inserts.


In [6]:
# ── Status — re-run this cell at any time ────────────────────────────────────
if bg['error']:
    print('❌ Error:', bg['error'])
elif bg['done']:
    elapsed = bg['t_end'] - bg['t_start']
    results = bg['results']
    print(f'✅ Done in {elapsed/60:.1f} min  ({len(results)} models returned)')
    display(results[['combination','r2','adj_r2','q2','mae','rmsd']].head(5))
elif bg['t_start']:
    so_far = time.perf_counter() - bg['t_start']
    print(f'⏳ Running… {so_far/60:.1f} min elapsed')
else:
    print('⏳ Not started yet.')

⏳ Running… 175.1 min elapsed


In [7]:
import sqlite3
import pandas as pd
from pathlib import Path

# Change this to your database path
DB_PATH = Path(r"C:\Users\edens\Desktop\DescriPytor\DescriPyTor-main\MolFeatures\M3_modeler\runs\doyle_features1_new_output_linear_20260629\db\results_doyle_features1_new.db")

# Output folder
OUT_DIR = Path("csv_output")
OUT_DIR.mkdir(exist_ok=True)

# Connect to the SQLite database
conn = sqlite3.connect(DB_PATH)

# Get all table names
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

print("Tables found:")
print(tables)

# Export each table to a separate CSV
for table_name in tables["name"]:
    df = pd.read_sql_query(f'SELECT * FROM "{table_name}"', conn)
    out_file = OUT_DIR / f"{table_name}.csv"
    df.to_csv(out_file, index=False)
    print(f"Saved: {out_file}")

conn.close()

Tables found:
                 name
0  regression_results
1     sqlite_sequence


ModuleNotFoundError: No module named 'pandas.io.formats.csvs'

In [12]:
import sqlite3
from pathlib import Path

DB_PATH = Path(r"C:\Users\edens\Desktop\DescriPytor\DescriPyTor-main\MolFeatures\M3_modeler\runs\doyle_features1_new_output_linear_20260629\db\results_doyle_features1_new.db")

# Safer read-only URI
db_uri = DB_PATH.resolve().as_uri() + "?mode=ro&immutable=1"

conn = sqlite3.connect(db_uri, uri=True, timeout=60)
cur = conn.cursor()

def quote_sql_name(name):
    return '"' + name.replace('"', '""') + '"'

cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = [row[0] for row in cur.fetchall()]

print("\nTables found:")
for table in tables:
    print(" ", table)

print("\nColumns by table:")
for table in tables:
    print("\n" + "=" * 80)
    print(f"TABLE: {table}")
    print("=" * 80)

    cur.execute(f"PRAGMA table_info({quote_sql_name(table)});")
    columns = cur.fetchall()

    for col in columns:
        cid, name, col_type, notnull, default_value, pk = col
        print(f"{cid:3d} | {name:35s} | {col_type}")

conn.close()


Tables found:
  regression_results
  sqlite_sequence

Columns by table:

TABLE: regression_results
  0 | id                                  | INTEGER
  1 | combination                         | TEXT
  2 | r2                                  | REAL
  3 | adj_r2                              | REAL
  4 | q2                                  | REAL
  5 | mae                                 | REAL
  6 | rmsd                                | REAL
  7 | threshold                           | REAL
  8 | model                               | TEXT
  9 | predictions                         | TEXT

TABLE: sqlite_sequence
  0 | name                                | 
  1 | seq                                 | 


In [ ]:
import sqlite3
import csv
from pathlib import Path

# DB_PATH = Path(r"C:\Users\edens\Downloads\your_file.db")
OUT_CSV = Path(r"C:\Users\edens\Downloads\regression_results_r2_below_0_77.csv")

R2_THRESHOLD = 0.77
Q2_THRESHOLD = 0.70
CHUNK_SIZE = 10000

db_uri = DB_PATH.resolve().as_uri() + "?mode=ro&immutable=1"
conn = sqlite3.connect(db_uri, uri=True, timeout=60)
cur = conn.cursor()

query = """
SELECT
    id,
    combination,
    r2,
    adj_r2,
    q2,
    mae,
    rmsd,
    threshold,
    model
FROM regression_results
WHERE r2 >= ?
  AND q2 >= ?
ORDER BY q2 DESC, r2 DESC
"""

cur.execute(
    """
    SELECT COUNT(*)
    FROM regression_results
    WHERE r2 >= ?
      AND q2 >= ?
    """,
    (R2_THRESHOLD, Q2_THRESHOLD)
)

n_rows = cur.fetchone()[0]
print(f"Rows with r2 >= {R2_THRESHOLD} and q2 >= {Q2_THRESHOLD}: {n_rows}")

cur.execute(query, (R2_THRESHOLD, Q2_THRESHOLD))

with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    headers = [desc[0] for desc in cur.description]
    writer.writerow(headers)

    total = 0
    while True:
        rows = cur.fetchmany(CHUNK_SIZE)
        if not rows:
            break

        writer.writerows(rows)
        total += len(rows)
        print(f"Exported {total}/{n_rows}")

conn.close()

print("Saved to:")
print(OUT_CSV)

Rows with r2 >= 0.77 and q2 >= 0.77: 19
Exported 19/19
Saved to:
C:\Users\edens\Downloads\regression_results_r2_below_0_77.csv


In [ ]:
# Block until done (run when you want to wait for the result)
search_thread.join()
results = bg['results']
print('Done. Models returned:', len(results))

## 5. Inspect results

In [ ]:
metric_cols  = ['combination', 'r2', 'adj_r2', 'q2', 'mae', 'rmsd']
display_cols = [c for c in metric_cols if c in results.columns]
display(results[display_cols].head(20))

In [ ]:
best = results.sort_values('q2', ascending=False).iloc[0]
print('Best combination:', best['combination'])
print(f"  R²      = {best.get('r2',     float('nan')):.4f}")
print(f"  adj-R²  = {best.get('adj_r2', float('nan')):.4f}")
print(f"  Q²(LOO) = {best.get('q2',     float('nan')):.4f}")
print(f"  MAE     = {best.get('mae',    float('nan')):.4f}")
print(f"  RMSD    = {best.get('rmsd',   float('nan')):.4f}")

## 6. Predict on the left-out set (L10, L27, L21, L18)

In [ ]:
from modeling import _normalize_combination_to_columns
from sklearn.metrics import mean_absolute_error, r2_score

best_cols = _normalize_combination_to_columns(best['combination'])
y_leftout = regression_model1.leftout_target_vector.to_numpy()

preds, errors = regression_model1.predict_for_leftout(
    X=regression_model1.leftout_samples[best_cols],
    y=regression_model1.leftout_target_vector,
    X_train=regression_model1.features_df[best_cols],
    y_train=regression_model1.target_vector,
)

leftout_df = pd.DataFrame({
    'molecule': leave_out,
    'y_true':   y_leftout,
    'y_pred':   preds,
    'error':    errors,
})
print(f'Left-out MAE : {mean_absolute_error(y_leftout, preds):.4f}')
print(f'Left-out R²  : {r2_score(y_leftout, preds):.4f}')
display(leftout_df)

## 7. Optional: drop outlier rows and re-run

In [ ]:
# drop1_nob = ['L20b', 'L22b', 'L23b', 'L24b', 'L12']
# regression_model1.drop_rows(drop1_nob)
# results2 = regression_model1.search_models(top_n=20, n_jobs=N_CORES, threshold=0.77)
# results2.head(10)

## 8. Reload saved results from disk

In [ ]:
saved = regression_model1.load_results(sort_key='adj_r2', ascending=False)
for name, df_saved in saved.items():
    print(f'\n── {name} ({len(df_saved)} rows) ──')
    display(df_saved.head(10))